In [ ]:
import torch
import numpy as np
import rasterio
from rasterio.windows import Window
import matplotlib.pyplot as plt
from transformers import AutoImageProcessor, SegformerForSemanticSegmentation
from tqdm import tqdm
image_path = "/home/ubuntu/work/satellite_data/digital_orthophoto_nrw/2021/2021/dop10rgbi_32_280_5652_1_nw_2021.tif"
device = 'cuda' if torch.cuda.is_available() else 'cpu'



In [ ]:
# online
model_id = "restor/tcd-segformer-mit-b2"
processor = AutoImageProcessor.from_pretrained(model_id)
model = SegformerForSemanticSegmentation.from_pretrained(model_id).to(device)


In [ ]:
# offline
model_path = "./local_tcd-segformer_local"
processor = AutoImageProcessor.from_pretrained(model_path, local_files_only=True)
model = SegformerForSemanticSegmentation.from_pretrained(model_path, local_files_only=True).to(device)

In [ ]:
def read_rgb(src, window=None, scale_16bit=True):
  # read rgb bands in satellite img.
  img = src.read([1,2,3], window=window)
  # Move channel axis to last dimension: (H, W, 3)
  img = np.moveaxis(img, 0, -1)
  if scale_16bit and img.dtype == np.uint16:
    img = (img / 256).astype(np.uint8)
  return img

In [ ]:
patch_size = 512

with rasterio.open(image_path) as src:
    # Sliced Inference
    h, w = src.height, src.width
    full_mask = np.zeros((h, w), dtype=np.uint8)
    # generate all windows covering the image with stride = 512
    windows = [(Window(x, y, min(patch_size, w - x), min(patch_size, h - y)), x, y,)
                for y in range(0, h, patch_size)
                for x in range(0, w, patch_size)
                ]

    # testing prints
    print(f"starting tiled inference on {image_path}")
    # processing of each patches
    for window, x, y in tqdm(windows, desc="Inference...", unit="patch"):
        patch = read_rgb(src, window=window)
        # run model inference
        inputs = processor(images=patch, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
        #  upsampling back to full mask
        mask = torch.nn.functional.interpolate(logits, size=(window.height, window.width), mode="bilinear")
        full_mask[y: y + window.height, x: x + window.width] = (mask.argmax(dim=1)[0].cpu().numpy() == 1).astype(np.uint8)

    full_img = read_rgb(src, window=None)

In [ ]:
# region to plot among the 10000x10000 image
x_start, x_end = 2000, 3000  
y_start, y_end = 3000, 4000 


# Crop mask and img accordingly
crop_img = full_img[y_start:y_end, x_start:x_end]
crop_mask = full_mask[y_start:y_end, x_start:x_end]

# Create overlay for cropped region 
overlay = crop_img.copy()
overlay[crop_mask == 1] = [0, 255, 0] # ==> positives represented as green
crop_result = (crop_img * 0.7 + overlay * 0.3).astype(np.uint8)

# Plot region 
fig = plt.figure(figsize=(9, 9))
ax = plt.Axes(fig, [0., 0., 1., 1.])
ax.set_axis_off()
fig.add_axes(ax)
ax.imshow(crop_result, interpolation='nearest')
# fig.savefig("cropped_tree_detection.png", dpi=1000, bbox_inches='tight', pad_inches=0)
# plt.close(fig)
plt.show()

## Convert to GeoJSON

In [ ]:
import geopandas as gpd
from rasterio.features import shapes
from shapely.geometry import shape
from rasterio.transform import from_bounds


# model_inference_func unused for now, as mask defined above.
def bbox_to_tree_geojson(bbox_coords, mask, model_inference_func=None):
    min_lon, min_lat, max_lon, max_lat = bbox_coords

    """
    mask = model_inference_func(img, ...) 
    """


    height, width = mask.shape
    
    transform = from_bounds(min_lon, min_lat, max_lon, max_lat, width, height)
    
    results = []
    for geom, value in shapes(mask, mask=(mask == 1), transform=transform):
        results.append({
            "geometry": shape(geom),
            "properties": {"class": "tree", "area_m2": None}
        })
    
    if not results:
        return {"type": "FeatureCollection", "features": []}
    
    gdf = gpd.GeoDataFrame.from_features(results, crs="EPSG:4326")
    gdf_projected = gdf.to_crs("EPSG:3857")
    gdf["area_m2"] = gdf_projected.geometry.area
    
    return gdf

result = bbox_to_tree_geojson((7.615, 51.955, 7.625, 51.965), full_mask)

In [ ]:
result